# Qwen3 Embedding + Answer API on Kaggle

This notebook loads two models in one Kaggle session:

1. Qwen3-Embedding-4B on GPU 0 for dense document/query vectors.
2. Qwen3-4B in 4-bit on GPU 1 for context-grounded answers.

Current limits: embedding inputs are capped at 8192 tokens and return the full 2560-dimensional vector; answer prompts plus generated output are capped at 32768 tokens, with 512 generated tokens by default.

It exposes authenticated POST endpoints through one ngrok HTTPS tunnel: /v1/embeddings, /v1/generate, and the legacy /v1/answer route. The retrieval app builds the final prompt and policy locally, then sends that prompt to /v1/generate.

Before running: enable Internet, select GPU T4 x2, and add Kaggle Secrets named NGROK_AUTHTOKEN and RAG_API_KEY. Do not paste either secret into a cell or print it. The ngrok URL is temporary and works only while this notebook session is running.

In [ ]:
%pip install -q -U "transformers>=4.51.0,<5" "sentence-transformers>=3.4.0,<6" "accelerate>=1.3.0,<2" "bitsandbytes>=0.46.1,<1" "fastapi>=0.110,<1" "uvicorn>=0.27,<1" "ngrok>=1.4,<2"

In [ ]:
import gc
import torch
import numpy as np

if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running this notebook.")

GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT < 2:
    raise RuntimeError(
        f"Found {GPU_COUNT} GPU(s). Select GPU T4 x2 so both models can stay loaded at once."
    )

for gpu_index in range(GPU_COUNT):
    props = torch.cuda.get_device_properties(gpu_index)
    memory_gib = props.total_memory / (1024 ** 3)
    print(f"GPU {gpu_index}: {props.name} | {memory_gib:.1f} GiB")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

## 1. Nạp Qwen3-Embedding-4B

Dùng FP16 và batch nhỏ để giữ mức dùng VRAM ổn định. Query được encode với prompt query tích hợp của Sentence Transformers; document dùng prompt mặc định.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-4B"
EMBEDDING_DEVICE = "cuda:0"
MAX_EMBEDDING_TOKENS = 8192
EMBEDDING_DIMENSION = 2560
EMBEDDING_BATCH_SIZE = 1

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=EMBEDDING_DEVICE,
    model_kwargs={"dtype": torch.float16},
    tokenizer_kwargs={"padding_side": "left"},
)
embedding_model.max_seq_length = MAX_EMBEDDING_TOKENS
actual_dimension = embedding_model.get_sentence_embedding_dimension()
if actual_dimension != EMBEDDING_DIMENSION:
    raise ValueError(f"Expected {EMBEDDING_DIMENSION} dimensions, got {actual_dimension}")


def embed_texts(texts, is_query=False):
    texts = [texts] if isinstance(texts, str) else list(texts)
    encode_kwargs = {
        "batch_size": EMBEDDING_BATCH_SIZE,
        "normalize_embeddings": True,
        "show_progress_bar": False,
        "convert_to_numpy": True,
    }
    if is_query:
        encode_kwargs["prompt_name"] = "query"
    vectors = embedding_model.encode(texts, **encode_kwargs)
    return np.asarray(vectors, dtype=np.float32)


print("Embedding model ready on", EMBEDDING_DEVICE)
print("Maximum input tokens:", embedding_model.max_seq_length)
print("Output dimensions:", actual_dimension)

In [ ]:
# Quick check: one document vector and one query vector.
demo_document = "Thông báo minh họa: Hệ thống sẽ bảo trì từ 23:00 ngày 10/11 đến 02:00 ngày 11/11."
demo_query = "Hệ thống bảo trì trong khung giờ nào?"

demo_document_vector = embed_texts([demo_document], is_query=False)
demo_query_vector = embed_texts([demo_query], is_query=True)

assert demo_document_vector.shape == demo_query_vector.shape
print("Document vector:", demo_document_vector.shape)
print("Query vector:", demo_query_vector.shape)
print("Cosine similarity:", float(demo_query_vector @ demo_document_vector.T))

## 2. Load Qwen3-4B for answer generation

Qwen3-4B runs in NF4 4-bit on GPU 1 with FP16 compute. Thinking mode is disabled. The notebook uses a conservative total prompt-plus-output budget of 32768 tokens and defaults to at most 512 generated tokens; the model card documents native 32768-token context (without YaRN).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

ANSWER_MODEL_NAME = "Qwen/Qwen3-4B"
ANSWER_GPU_INDEX = 1
ANSWER_DEVICE = f"cuda:{ANSWER_GPU_INDEX}"
ANSWER_CONTEXT_LIMIT_TOKENS = 32768
DEFAULT_ANSWER_MAX_NEW_TOKENS = 512
MAX_ANSWER_MAX_NEW_TOKENS = 768
MAX_GENERATE_NEW_TOKENS = 4096

answer_tokenizer = AutoTokenizer.from_pretrained(ANSWER_MODEL_NAME)
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

answer_model = AutoModelForCausalLM.from_pretrained(
    ANSWER_MODEL_NAME,
    quantization_config=quantization_config,
    dtype=torch.float16,
    device_map={"": ANSWER_GPU_INDEX},
    low_cpu_mem_usage=True,
)
answer_model.eval()
print("Answer model ready on", ANSWER_DEVICE)
print("Total input + output token budget:", ANSWER_CONTEXT_LIMIT_TOKENS)
print("Default max_new_tokens:", DEFAULT_ANSWER_MAX_NEW_TOKENS)
print("Embedding model remains loaded on", EMBEDDING_DEVICE)

In [ ]:
def answer_with_context(
    question,
    contexts,
    max_new_tokens=DEFAULT_ANSWER_MAX_NEW_TOKENS,
):
    contexts = [contexts] if isinstance(contexts, str) else list(contexts)
    context_block = "\n\n".join(
        f"[{index}] {context}" for index, context in enumerate(contexts, start=1)
    )
    messages = [
        {
            "role": "system",
            "content": (
                "B?n l? tr? l? h?i ??p d?a tr?n t?i li?u. Ch? d?ng th?ng tin trong context. "
                "N?u context kh?ng ?? ?? tr? l?i, h?y n?i r? l? ch?a t?m th?y th?ng tin. "
                "Kh?ng t? suy ?o?n. D?n ngu?n b?ng s? trong ngo?c vu?ng, v? d? [1]."
            ),
        },
        {
            "role": "user",
            "content": f"CONTEXT:\n{context_block}\n\nC?U H?I:\n{question}",
        },
    ]
    model_inputs = answer_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    )
    input_device = answer_model.get_input_embeddings().weight.device
    model_inputs = {key: value.to(input_device) for key, value in model_inputs.items()}
    prompt_length = model_inputs["input_ids"].shape[-1]
    if prompt_length + max_new_tokens > ANSWER_CONTEXT_LIMIT_TOKENS:
        raise ValueError(
            f"Prompt plus output needs {prompt_length + max_new_tokens} tokens; "
            f"limit is {ANSWER_CONTEXT_LIMIT_TOKENS}. Send fewer or shorter contexts."
        )
    with torch.inference_mode():
        generated = answer_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    answer_tokens = generated[0][prompt_length:]
    return answer_tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()


def generate_from_prompt(
    prompt,
    max_new_tokens=DEFAULT_ANSWER_MAX_NEW_TOKENS,
    temperature=0.1,
):
    rendered = answer_tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    model_inputs = answer_tokenizer(
        [rendered],
        return_tensors="pt",
    )
    input_device = answer_model.get_input_embeddings().weight.device
    model_inputs = {key: value.to(input_device) for key, value in model_inputs.items()}
    prompt_length = model_inputs["input_ids"].shape[-1]
    if prompt_length + max_new_tokens > ANSWER_CONTEXT_LIMIT_TOKENS:
        raise ValueError(
            f"Prompt plus output needs {prompt_length + max_new_tokens} tokens; "
            f"limit is {ANSWER_CONTEXT_LIMIT_TOKENS}."
        )
    generation_options = {"do_sample": temperature > 0}
    if temperature > 0:
        generation_options.update(temperature=temperature, top_p=0.8, top_k=20)
    with torch.inference_mode():
        generated = answer_model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=answer_tokenizer.eos_token_id,
            **generation_options,
        )
    answer_tokens = generated[0][prompt_length:]
    answer = answer_tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()
    if not answer:
        raise RuntimeError("Model returned an empty generation.")
    return answer

In [ ]:
# Smoke example with a synthetic context; replace it with retrieved chunks when ready.
demo_answer = answer_with_context(
    question="Hệ thống sẽ bảo trì trong khung giờ nào?",
    contexts=[demo_document],
)
print(demo_answer)

## API usage and model settings

- POST /v1/embeddings accepts {"texts": ["..."], "is_query": true|false}; returns normalized 2560-dimensional dense vectors. Each input is truncated to 8192 tokens by the embedding model.
- POST /v1/generate accepts a complete prompt and policy built by the local PromptBuilder; output defaults to 512 tokens, with a request limit of 4096.
- POST /v1/answer remains available for the earlier question-plus-context client; answer output defaults to 512 tokens, with a request limit of 768.
- For answers, the prompt token count plus generated tokens must stay within 32768; oversized requests return HTTP 413.
- Both routes require Authorization: Bearer <RAG_API_KEY>. OpenAPI docs are disabled on the public tunnel.
- The API does not perform retrieval. The caller retrieves from Qdrant between the embedding and answer calls.
- Kaggle sessions are temporary; the tunnel and URL stop when the session ends.

## 3. Start the authenticated API and ngrok tunnel

Store NGROK_AUTHTOKEN and RAG_API_KEY in Kaggle Add-ons > Secrets, and attach both secrets to this notebook. Use a long random RAG_API_KEY and keep the same value in the local client environment.

In [ ]:
from kaggle_secrets import UserSecretsClient
from fastapi import Depends, FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
import hmac
import os
import threading
import time
import uvicorn
import ngrok

user_secrets = UserSecretsClient()
try:
    os.environ["NGROK_AUTHTOKEN"] = user_secrets.get_secret("NGROK_AUTHTOKEN")
    RAG_API_KEY = user_secrets.get_secret("RAG_API_KEY")
except Exception as exc:
    raise RuntimeError(
        "Add and attach NGROK_AUTHTOKEN and RAG_API_KEY in Kaggle Add-ons > Secrets."
    ) from exc

if not os.environ["NGROK_AUTHTOKEN"] or not RAG_API_KEY:
    raise RuntimeError("Both Kaggle secrets must have non-empty values.")

embedding_lock = threading.Lock()
answer_lock = threading.Lock()
app = FastAPI(
    title="Qwen3 RAG Model API",
    docs_url=None,
    redoc_url=None,
    openapi_url=None,
)

def require_api_key(authorization: str | None = Header(default=None)):
    expected = f"Bearer {RAG_API_KEY}"
    if authorization is None or not hmac.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail="Unauthorized")

class EmbeddingsRequest(BaseModel):
    texts: list[str] = Field(min_length=1, max_length=16)
    is_query: bool = False

class AnswerRequest(BaseModel):
    question: str = Field(min_length=1, max_length=6000)
    contexts: list[str] = Field(min_length=1, max_length=8)
    max_new_tokens: int = Field(default=DEFAULT_ANSWER_MAX_NEW_TOKENS, ge=1, le=MAX_ANSWER_MAX_NEW_TOKENS)

class GenerateRequest(BaseModel):
    prompt: str = Field(min_length=1, max_length=60000)
    max_new_tokens: int = Field(default=DEFAULT_ANSWER_MAX_NEW_TOKENS, ge=1, le=MAX_GENERATE_NEW_TOKENS)
    temperature: float = Field(default=0.1, ge=0, le=2)

@app.get("/health")
def health(_=Depends(require_api_key)):
    return {
        "status": "ok",
        "embedding_model": EMBEDDING_MODEL_NAME,
        "answer_model": ANSWER_MODEL_NAME,
    }

@app.post("/v1/embeddings")
def create_embeddings(payload: EmbeddingsRequest, _=Depends(require_api_key)):
    if any(not text.strip() or len(text) > 20000 for text in payload.texts):
        raise HTTPException(status_code=413, detail="Each text must be 1-20000 characters.")
    with embedding_lock:
        vectors = embed_texts(payload.texts, is_query=payload.is_query)
    return {
        "model": EMBEDDING_MODEL_NAME,
        "dimension": int(vectors.shape[1]),
        "embeddings": vectors.tolist(),
    }

@app.post("/v1/answer")
def create_answer(payload: AnswerRequest, _=Depends(require_api_key)):
    if any(not context.strip() for context in payload.contexts):
        raise HTTPException(status_code=422, detail="Contexts must not be empty.")
    if sum(len(context) for context in payload.contexts) > 50000:
        raise HTTPException(status_code=413, detail="Combined contexts must be at most 50000 characters.")
    try:
        with answer_lock:
            answer = answer_with_context(
                payload.question,
                payload.contexts,
                max_new_tokens=payload.max_new_tokens,
            )
    except ValueError as exc:
        raise HTTPException(status_code=413, detail=str(exc)) from exc
    return {"model": ANSWER_MODEL_NAME, "answer": answer}

@app.post("/v1/generate")
def create_generation(payload: GenerateRequest, _=Depends(require_api_key)):
    if not answer_lock.acquire(blocking=False):
        raise HTTPException(status_code=429, detail="Generation in progress")
    try:
        text = generate_from_prompt(
            payload.prompt,
            max_new_tokens=payload.max_new_tokens,
            temperature=payload.temperature,
        )
        return {"model": ANSWER_MODEL_NAME, "text": text}
    except ValueError as exc:
        raise HTTPException(status_code=413, detail=str(exc)) from exc
    except Exception:
        raise HTTPException(status_code=503, detail="Generation unavailable") from None
    finally:
        answer_lock.release()

print("Authenticated API routes are ready.")

In [ ]:
api_config = uvicorn.Config(
    app,
    host="127.0.0.1",
    port=8000,
    log_level="warning",
)
api_server = uvicorn.Server(api_config)
api_thread = threading.Thread(target=api_server.run, daemon=True)
api_thread.start()

for _ in range(30):
    if api_server.started:
        break
    time.sleep(1)
if not api_server.started:
    raise RuntimeError("Uvicorn did not start on port 8000.")

ngrok_forwarder = await ngrok.forward(
    "localhost:8000",
    authtoken_from_env=True,
)
PUBLIC_API_URL = ngrok_forwarder.url()
print("API URL:", PUBLIC_API_URL)
print("Routes: POST /v1/embeddings, POST /v1/generate, POST /v1/answer, GET /health")
print("Use Authorization: Bearer <your RAG_API_KEY> for every request.")

## Call it from the local retrieval code

Set RAG_API_URL to the printed ngrok base URL and RAG_API_KEY to the same secret value. Set QWEN_API_URL to <RAG_API_URL>/v1/generate and QWEN_API_TOKEN to the same key. The local flow is: embed query through Kaggle -> search local Qdrant -> build the governed prompt locally -> send it to /v1/generate.

## Call it from Windows PowerShell

Copy the current ngrok URL printed above. Set the same RAG_API_KEY value you stored in Kaggle Secrets. Do not save the real key in this notebook.

```powershell
$env:RAG_API_URL = "<current ngrok URL>"
$env:RAG_API_KEY = "<your secret value>"
$env:QWEN_API_URL = "<current ngrok URL>/v1/generate"
$env:QWEN_API_TOKEN = "<your secret value>"
```

Then call the API from your local retrieval code:

```python
import os
import requests

api_url = os.environ["RAG_API_URL"].rstrip("/")
headers = {"Authorization": f"Bearer {os.environ['RAG_API_KEY']}"}

embedded = requests.post(
    f"{api_url}/v1/embeddings",
    headers=headers,
    json={"texts": ["C?u h?i c?n t?m ki?m"], "is_query": True},
    timeout=120,
)
embedded.raise_for_status()
query_vector = embedded.json()["embeddings"][0]

# Search local Qdrant with query_vector, then build this prompt with PromptBuilder.
prompt = "SYSTEM\n<policy>\nCONTEXT\n[chunk-01] retrieved text\n\nQUESTION\nuser question\n\nANSWER\n"
generated = requests.post(
    f"{api_url}/v1/generate",
    headers=headers,
    json={
        "prompt": prompt,
        "max_new_tokens": 512,
        "temperature": 0.1,
    },
    timeout=300,
)
generated.raise_for_status()
print(generated.json()["text"])
```